# Plantilla de algoritmo. Parte 3 (Problema del Viajante)

**Cómo usarla:** copia este archivo como `parte3/<algoritmo>_<tunombre>.ipynb` (por ejemplo, `parte3/fuerza_bruta_pablo.ipynb`) y trabaja solo **debajo de la línea ✂️**. Lo de arriba (preparación) es el mismo código de la Parte 2 del trabajo y sirve para cargar las versiones del grafo: **no lo modifiques**. La guía completa está en `CONTRIBUTING.md`.

> ⚙️ El notebook se ejecuta desde su carpeta (`parte3/`): la primera celda sube a la raíz del proyecto para que las rutas (`data/...`) funcionen.

In [1]:
# Sube a la raíz del proyecto para que las rutas (data/ciudades.csv, data/versiones/...) funcionen
import os
from pathlib import Path
if Path.cwd().name == "parte3":
    os.chdir("..")

In [2]:
# Librerías que se usan en todo el notebook
import csv                       # leer y escribir tablas en formato CSV
import itertools                 # recorrer todas las combinaciones de parámetros
import unicodedata               # quitar acentos de los nombres (Cataluña -> cataluna)
from pathlib import Path         # rutas de archivos

import networkx as nx            # crear y analizar grafos (la misma librería de las prácticas)
import numpy as np               # tablas de números (las tablas de distancias)
from scipy.sparse import csr_matrix                 # matriz del grafo para el algoritmo de Dijkstra
from scipy.sparse.csgraph import dijkstra           # caminos mínimos desde varios orígenes
from scipy.spatial import cKDTree                   # buscar rápido el nodo más cercano a un punto

from tsp_utils import dibujar_red  # función de dibujo de la red de carreteras, comentada en tsp_utils.py

In [3]:
# ── Parámetros de las versiones (editables) ──
X_KM  = 10             # una ciudad a más de X_KM km de la red se deja fuera de la versión
N_MIN = 4              # mínimo de ciudades para una versión por comunidad autónoma
RECALCULAR = False     # True: vuelve a calcular las 16 tablas (necesita el grafo G de la Parte 1)
CARPETA = Path("data/versiones")     # aquí se guardan las tablas ya calculadas

# Qué tipos de vía (`highway`) puede usar cada versión de carreteras
CARRETERAS = {
    "completo":             ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link", "secondary", "secondary_link"],
    "autovias_principales": ["motorway", "motorway_link", "trunk", "trunk_link",
                             "primary", "primary_link"],
    "solo_autovias":        ["motorway", "motorway_link"],
    "sin_autovias":         ["trunk", "trunk_link", "primary", "primary_link",
                             "secondary", "secondary_link"],
}
PESOS = {"km": ("length", 1000), "tiempo": ("travel_time", 1)}   # peso -> (atributo de la arista, divisor)
CONJUNTOS = ["capitales", "capitales_y_100k"]                    # qué ciudades se visitan
GENERALES = ["multigrafo_dirigido", "grafo_dirigido",
             "multigrafo_no_dirigido", "grafo_simple_no_dirigido"]

In [4]:
# Lee la lista de ciudades (nombre, comunidad, si es capital de provincia, coordenadas...)
with open("data/ciudades.csv", encoding="utf-8") as f:
    CIUDADES = list(csv.DictReader(f))
for c in CIUDADES:
    c["lat"], c["lon"] = float(c["lat"]), float(c["lon"])
    c["capital"] = c["capital_provincia"] == "True"

def slug(texto):
    # "Castilla y León" -> "castilla_y_leon" (sin acentos ni espacios, para usarlo como nombre)
    sin_acentos = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return sin_acentos.lower().replace(" ", "_").replace("-", "_")

AMBITOS = ["espana"] + sorted({slug(c["comunidad"]) for c in CIUDADES})

def ciudades_de(nodos, ambito):
    # Nombres de las ciudades de un conjunto: nodos = "capitales" o "capitales_y_100k";
    # ambito = "espana" o una comunidad (p. ej. "galicia")
    return [c["nombre"] for c in CIUDADES
            if (c["capital"] or nodos == "capitales_y_100k")
            and (ambito == "espana" or slug(c["comunidad"]) == ambito)]

print(f"{len(CIUDADES)} ciudades | {len(ciudades_de('capitales', 'espana'))} capitales | ámbitos: {len(AMBITOS) - 1} comunidades")
print("Capitales de Galicia:", ciudades_de("capitales", "galicia"))

75 ciudades | 47 capitales | ámbitos: 15 comunidades
Capitales de Galicia: ['A Coruña', 'Ourense', 'Lugo', 'Pontevedra']


In [5]:
def nombre_tabla(aristas, peso, dirigido):
    return CARPETA / f"{aristas}__{peso}__{'dirigido' if dirigido else 'no_dirigido'}.csv"

def guardar_tabla(D, nombres, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        np.savetxt(f, D, delimiter=",", header=",".join(nombres), comments="", fmt="%.3f")

def leer_tabla(aristas, peso, dirigido):
    ruta = nombre_tabla(aristas, peso, dirigido)
    with open(ruta, encoding="utf-8") as f:
        nombres = f.readline().strip().split(",")        # primera fila: nombres de las ciudades
    return np.loadtxt(ruta, delimiter=",", skiprows=1, encoding="utf-8"), nombres

def escribir_csv(ruta, filas, campos):
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=campos)
        w.writeheader()
        w.writerows(filas)

In [6]:
UNIDAD = {"km": "km", "tiempo": "segundos"}

def cargar_version(aristas, nodos, ambito, peso, general):
    # aristas: "completo" | "autovias_principales" | "solo_autovias" | "sin_autovias"
    # nodos:   "capitales" | "capitales_y_100k"       ambito: "espana" | "galicia" | "andalucia" ...
    # peso:    "km" | "tiempo"                        general: uno de GENERALES
    # (multigrafo y grafo simple dan el mismo grafo de ciudades: no hay aristas repetidas)
    dirigido = "no_dirigido" not in general
    D, nombres = leer_tabla(aristas, peso, dirigido)
    elegidas = [c for c in ciudades_de(nodos, ambito) if c in nombres]    # sin las que quedaron fuera
    if ambito != "espana" and len(elegidas) < N_MIN:
        raise ValueError(f"Solo hay {len(elegidas)} ciudades (mínimo {N_MIN}) en {ambito}")
    K = nx.DiGraph() if dirigido else nx.Graph()
    for c in CIUDADES:
        if c["nombre"] in elegidas:
            K.add_node(c["nombre"], lat=c["lat"], lon=c["lon"])
    for p, a in enumerate(elegidas):
        for q, b in enumerate(elegidas):
            if p != q and (dirigido or p < q):                            # no dirigido: una arista por par
                K.add_edge(a, b, peso=float(D[nombres.index(a), nombres.index(b)]))
    K.graph.update(aristas=aristas, nodos=nodos, ambito=ambito, peso=peso,
                   unidad=UNIDAD[peso], general=general,
                   fuera=[c for c in ciudades_de(nodos, ambito) if c not in nombres])
    return K

In [7]:
# [AYUDA COMPARTIDA] Funciones comunes a todos los algoritmos
import time
import tracemalloc

def coste_ruta(K, ruta):
    # Coste total del ciclo: suma de las distancias entre ciudades consecutivas y de la última a la primera
    return sum(K[ruta[i]][ruta[(i + 1) % len(ruta)]]["peso"] for i in range(len(ruta)))

def validar_ruta(K, ruta):
    # Comprueba que la ruta visita cada ciudad de K exactamente una vez
    assert len(ruta) == K.number_of_nodes() and set(ruta) == set(K.nodes), "la ruta no visita cada ciudad una vez"

def medir(algoritmo, K, memoria=False, **opciones):
    # Ejecuta algoritmo(K, **opciones), comprueba el resultado y devuelve cuánto ha tardado
    # (con memoria=True mide también el pico de memoria en una segunda ejecución, porque medirla ralentiza)
    t0 = time.perf_counter()
    ruta, coste = algoritmo(K, **opciones)
    segundos = time.perf_counter() - t0
    validar_ruta(K, ruta)
    assert abs(coste - coste_ruta(K, ruta)) < 1e-6 * max(1.0, abs(coste)), "el coste devuelto no coincide con el de la ruta"
    resultado = {"algoritmo": algoritmo.__name__, "ciudades": K.number_of_nodes(), "coste": round(coste, 3),
                 "unidad": K.graph.get("unidad"), "segundos": round(segundos, 4)}
    if memoria:
        tracemalloc.start()
        algoritmo(K, **opciones)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        resultado["memoria_mb"] = round(pico / 1e6, 2)
    return resultado

---
### ✂️ A PARTIR DE AQUÍ SE INTEGRA EN EL TRABAJO

Todo lo que escribas **debajo de esta línea** pasará al notebook principal (`trabajo_tsp.ipynb`, Parte 3). No modifiques esta celda.

### Vecino más cercano (voraz)

<div style="border-left:5px solid #4C72B0;color:#000;background:#EEF2FA;padding:10px 14px;margin:10px 0;border-radius:4px;">

**📚 TEORÍA**

Es la estrategia del viajero que no planifica: **en cada ciudad, ir a la más cercana que todavía no haya visitado**. No se mira el recorrido completo ni se vuelve atrás para corregir; se decide paso a paso y lo decidido ya no se toca. Es la traducción literal de un algoritmo **voraz** (*greedy*): tomar siempre la opción que parece mejor en ese instante. A cambio de su simplicidad se renuncia a la garantía de optimalidad: una elección magnífica ahora puede dejar aislada una ciudad lejana y arruinar el final del recorrido.

**Aplicado al problema del viajante.** Dado el grafo de ciudades `K` (completo, con `K[a][b]["peso"]` = coste de ir de `a` a `b`), se parte de una ciudad y, mientras queden ciudades sin visitar, se salta a la **más barata** de las pendientes **desde la ciudad actual**. Cuando no queda ninguna, se cierra el ciclo volviendo a la ciudad de partida. El resultado es un ciclo hamiltoniano (todas las ciudades una sola vez) construido sin mirar atrás. Como `K` es **completo**, siempre hay una ciudad pendiente a la que saltar y el ciclo se puede cerrar; con un grafo incompleto habría que comprobar además que quedan aristas.

**Pseudocódigo.**

<pre style="background:#fff;border:1px solid #ccd;padding:10px 14px;border-radius:4px;font-size:13px;">
VECINO_MÁS_CERCANO(K, inicio):
    si K no tiene ciudades: devolver ([], 0)
    inicio ← la ciudad pedida, o la primera de K si no se dio ninguna
    ruta ← [inicio] ;  pendientes ← todas las ciudades menos inicio ;  coste ← 0

    mientras queden ciudades en pendientes:
        actual ← última ciudad de ruta
        siguiente ← la ciudad de pendientes con menor K[actual][ciudad]["peso"]
                    (si hay empate, la primera por orden alfabético)
        coste ← coste + K[actual][siguiente]["peso"]
        ruta ← ruta + [siguiente] ;  quitar siguiente de pendientes

    coste ← coste + K[última][inicio]["peso"]      # la vuelta, que cierra el ciclo
    devolver (ruta, coste)
</pre>

**Complejidad.** **Tiempo: O(n²)** — hay `n−1` pasos y en cada uno se compara la ciudad actual con las pendientes (hasta `n−1`), con acceso O(1) al peso de cada arista; quitar la elegida de la lista añade otro recorrido O(n) por paso, que no cambia el orden. **Memoria: O(n)** adicional — solo la ruta y la lista de pendientes; el grafo ya ocupa O(n²) por ser completo, pero el algoritmo no construye ninguna tabla nueva. Compárese con el `O((n−1)!·n)` de la fuerza bruta: la diferencia entre milisegundos y siglos.

**Propiedades.**

- **Tipo:** **heurístico constructivo** (voraz). No es exacto ni tiene garantía de aproximación: construye una solución buena sin prometer nada.
- **¿Siempre encuentra el óptimo?** **No.** Es fácil construir casos en los que una arista barata al principio obliga a un salto carísimo al final. En las pruebas de abajo se mide cuánto se aleja del óptimo real.
- **¿Determinista?** **Sí**: no usa azar y los empates se resuelven por orden alfabético, así que la misma entrada devuelve siempre la misma ruta.
- **¿Garantía de calidad?** **Ninguna en general.** Incluso con distancias que cumplen la desigualdad triangular, el peor caso puede ser Θ(log n) veces el óptimo. Con un arranque razonable suele quedarse a un 1–15 %, pero un mal arranque puede superar el 40 % (se mide en las pruebas de abajo); por eso se usa como punto de partida para mejoras locales (2-opt).
- **¿Necesita distancias simétricas?** **No.** Solo lee `K[a][b]` en el sentido en que se viaja, así que funciona igual con `nx.DiGraph` (ida y vuelta distintas por los sentidos únicos) que con `nx.Graph`.
- **¿Depende de la ciudad de arranque?** **Sí**, y bastante: en un grafo dirigido el mismo conjunto de ciudades puede costar muy distinto según dónde se empiece. Es consecuencia de la voracidad, no un fallo; probar varios arranques (multiarranque) es una mejora barata, aunque el algoritmo que se entrega aquí es el vecino más cercano puro.
- **¿Hasta cuántas ciudades?** **Todas**: con 75 tarda menos de un milisegundo. El límite no es el tiempo, sino la calidad de la solución.
- **No es Dijkstra.** Dijkstra calcula el camino más corto entre dos puntos; esto construye un recorrido que pasa por todas las ciudades.


**💻 CÓDIGO**

La implementación mantiene la lista de ciudades pendientes y, en cada paso, se queda con la de menor peso desde la ciudad actual con `min(..., key=...)`. El desempate por `(peso, nombre)` hace que el resultado no dependa del orden interno del grafo y sea **reproducible**. El coste se acumula arista a arista, incluida la vuelta al inicio, sin recalcular la ruta entera. No se supone simetría: solo se consulta `K[a][b]` en el sentido del viaje.

In [8]:
def vecino_mas_cercano(K, inicio=None):
    # Vecino más cercano (voraz) para el problema del viajante: en cada paso va a la ciudad sin
    # visitar más barata desde la actual. Contrato: K es el grafo de ciudades (completo,
    # K[a][b]["peso"] = distancia de a a b) y devuelve (ruta, coste), donde `ruta` es la lista con
    # todas las ciudades, cada una una vez, en el orden en que se visitan (el viajante vuelve
    # después a la primera), y `coste` es el coste total del ciclo, incluida esa vuelta. No asume
    # simetría: sirve para K dirigido (nx.DiGraph) y no dirigido (nx.Graph) por igual.

    ciudades = list(K.nodes)

    # Casos triviales: sin ciudades o con una sola, el ciclo ya está determinado
    if not ciudades:
        return [], 0.0
    if inicio is None:
        inicio = ciudades[0]                       # por defecto, la primera ciudad del grafo
    elif inicio not in K:
        raise KeyError(f"La ciudad de inicio {inicio!r} no está en el grafo")

    # Lista de ciudades que quedan por visitar y ruta construida hasta ahora
    pendientes = [c for c in ciudades if c != inicio]
    ruta = [inicio]
    coste = 0.0
    if not pendientes:
        return ruta, coste                       # una sola ciudad: no hay nada que recorrer

    # Bucle voraz: n-1 pasos, uno por cada ciudad que falta
    while pendientes:
        actual = ruta[-1]

        # La más cercana de las pendientes; el desempate por nombre hace el resultado reproducible
        siguiente = min(pendientes, key=lambda c: (K[actual][c]["peso"], c))

        # Sumamos la arista elegida y avanzamos (la ciudad ya no se puede volver a visitar)
        coste += K[actual][siguiente]["peso"]
        ruta.append(siguiente)
        pendientes.remove(siguiente)

    # La vuelta desde la última ciudad a la de inicio, que cierra el ciclo
    coste += K[ruta[-1]][inicio]["peso"]
    return ruta, coste

**Prueba rápida.** Dos comprobaciones:

1. **Versión pequeña real:** las 6 ciudades de Galicia, empezando en Vigo, con `validar_ruta` y `medir` (tiempo y memoria).
2. **¿Cuánto se aleja del óptimo?** Se enumeran los `6! = 720` ciclos posibles con `coste_ruta` y se compara el mejor con el que devuelve el voraz. Es la referencia exacta para esta instancia.

In [ ]:
# ── 1. Versión pequeña real: Galicia (6 ciudades), empezando en Vigo ──
K = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="galicia", peso="km", general="grafo_dirigido")
ruta, coste = vecino_mas_cercano(K, inicio="Vigo")
validar_ruta(K, ruta)                               # comprueba que la ruta visita cada ciudad una vez
print(" → ".join(ruta), f"| coste: {coste:.1f} {K.graph['unidad']}")
print(medir(vecino_mas_cercano, K, memoria=True, inicio="Vigo"))

# ── 2. Referencia independiente: el mejor de los 6! = 720 ciclos posibles ──
# Cualquier ciclo se puede escribir empezando por cualquier ciudad, así que enumerar los 720
# ciclos equivale a comparar con el óptimo real de esta instancia (no solo con los que empiezan en Vigo).
optimo = min(coste_ruta(K, list(p)) for p in itertools.permutations(K.nodes))
print(f"\nóptimo de los 720 ciclos: {optimo:.3f} km | voraz desde Vigo: {coste:.3f} km "
      f"| diferencia: {coste - optimo:+.3f} km ({(coste - optimo) / optimo * 100:+.1f} %)")

Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Ourense | coste: 465.8 km
{'algoritmo': 'vecino_mas_cercano', 'ciudades': 6, 'coste': 465.817, 'unidad': 'km', 'segundos': 0.0, 'memoria_mb': 0.0}

óptimo de los 720 ciclos: 460.438 km | voraz desde Vigo: 465.817 km | diferencia: +5.379 km (+1.2 %)


**Calidad y dependencia del arranque.**

3. **Frente al óptimo, con más ciudades:** sobre Andalucía (13 ciudades) se toman subgrafos de 5 a 9 ciudades. Para cada uno se calcula el óptimo enumerando todos los ciclos y se ejecuta el voraz **desde cada una de sus ciudades**, para ver el mejor y el peor resultado posible. Así se mide a la vez la calidad y cuánto influye el arranque.
4. **Determinismo:** dos ejecuciones seguidas deben dar la misma ruta y el mismo coste.
5. **Ciudad de arranque:** el mismo grafo de Galicia recorrido desde cada una de sus ciudades, y el rango de costes al arrancar en cada una de las 75 ciudades de España. Sirve para ver cuánto pesa la primera decisión.

In [ ]:
# ── 3. Calidad: voraz frente al óptimo en instancias cada vez mayores ──
KAnd = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="andalucia", peso="km", general="grafo_dirigido")
ciudades = sorted(KAnd.nodes)

print(f"Andalucía: {KAnd.number_of_nodes()} ciudades disponibles")
print(f"{'ciudades':>9} | {'óptimo (km)':>12} | {'mejor voraz':>11} | {'peor voraz':>10} | {'gap peor':>8}")
for n in range(5, 10):
    Kn = KAnd.subgraph(ciudades[:n]).copy()          # subgrafo completo de las n primeras ciudades
    optimo_n = min(coste_ruta(Kn, list(p)) for p in itertools.permutations(Kn.nodes))
    # Voraz desde cada ciudad (orden alfabético, reproducible): mejor y peor resultado posible
    costes_n = [vecino_mas_cercano(Kn, inicio=ini)[1] for ini in sorted(Kn.nodes)]
    gap_peor = (max(costes_n) - optimo_n) / optimo_n * 100
    if abs(gap_peor) < 0.05:                         # evita imprimir "-0.0 %" por ruido de coma flotante
        gap_peor = 0.0
    print(f"{n:>9} | {optimo_n:>12.1f} | {min(costes_n):>11.1f} | {max(costes_n):>10.1f} | {gap_peor:>7.1f} %")

# ── 4. Determinismo: sin azar, dos ejecuciones dan exactamente lo mismo ──
ruta_a, coste_a = vecino_mas_cercano(K)
ruta_b, coste_b = vecino_mas_cercano(K)
assert ruta_a == ruta_b and abs(coste_a - coste_b) < 1e-9, "el algoritmo no es determinista"
print(f"\nDeterminista: dos ejecuciones idénticas ({coste_a:.3f} km) ✔")

# ── 5. ¿Cuánto importa la ciudad de arranque? ──
print("\nMismo grafo de Galicia, arrancando en cada ciudad:")
costes = {ini: round(vecino_mas_cercano(K, inicio=ini)[1], 1) for ini in sorted(K.nodes)}
for ini, c_ini in costes.items():
    print(f"  {ini:>24} → {c_ini:>7.1f} km")
mejor, peor = min(costes.values()), max(costes.values())
print(f"mejor arranque: {mejor:.1f} km ({', '.join(c for c in costes if costes[c] == mejor)})")
print(f"peor arranque:  {peor:.1f} km ({', '.join(c for c in costes if costes[c] == peor)}) "
      f"| diferencia: {(peor - mejor) / mejor * 100:.1f} %")

# En las 75 ciudades de España: rango de resultados según dónde se empiece (75 ejecuciones, < 0,1 s)
K75 = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="espana", peso="km", general="grafo_dirigido")
costes75 = {ini: vecino_mas_cercano(K75, inicio=ini)[1] for ini in K75.nodes}
coste_defecto = vecino_mas_cercano(K75)[1]
mejor75 = min(costes75, key=costes75.get)
peor75 = max(costes75, key=costes75.get)
print(f"\n75 ciudades: por defecto {coste_defecto:.0f} km | mejor arranque {costes75[mejor75]:.0f} km ({mejor75}) | "
      f"peor {costes75[peor75]:.0f} km ({peor75}) | media {sum(costes75.values()) / len(costes75):.0f} km")

Andalucía: 13 ciudades disponibles
 ciudades |  óptimo (km) | mejor voraz | peor voraz | gap peor
        5 |        991.3 |       991.3 |     1281.5 |    29.3 %
        6 |       1006.4 |      1006.4 |     1074.6 |     6.8 %
        7 |       1197.6 |      1197.6 |     1483.3 |    23.9 %
        8 |       1231.4 |      1231.4 |     1777.2 |    44.3 %
        9 |       1235.1 |      1235.1 |     1782.9 |    44.4 %

Determinista: dos ejecuciones idénticas (460.438 km) ✔

Mismo grafo de Galicia, arrancando en cada ciudad:
                  A Coruña →   460.4 km
                      Lugo →   460.4 km
                   Ourense →   488.7 km
                Pontevedra →   460.4 km
    Santiago de Compostela →   488.7 km
                      Vigo →   465.8 km
mejor arranque: 460.4 km (A Coruña, Lugo, Pontevedra)
peor arranque:  488.7 km (Ourense, Santiago de Compostela) | diferencia: 6.1 %

75 ciudades: por defecto 8102 km | mejor arranque 6965 km (Córdoba) | peor 8641 km (León) | media 77

**Escalabilidad.** El voraz se mide en todas las versiones de España (4 tipos de carreteras × 2 conjuntos de ciudades × dirigido/no dirigido), siempre arrancando en Madrid para que las filas sean comparables. `solo_autovias` tiene 3 ciudades menos (Salamanca, Badajoz y Cuenca quedan fuera por estar a más de 10 km de la red). Se añade además la misma versión con peso en tiempo, para comprobar que el algoritmo es indiferente a la unidad.

In [ ]:
# ── 6. Escalabilidad: todas las versiones de España, con 47 y 75 ciudades ──
print(f"{'aristas':>21} | {'peso':>4} | {'general':>24} | {'ciudades':>8} | {'coste':>10} | {'tiempo':>9} | {'memoria':>9}")
for aristas in ("completo", "autovias_principales", "sin_autovias", "solo_autovias"):
    for nodos in ("capitales", "capitales_y_100k"):
        for general in ("grafo_dirigido", "grafo_simple_no_dirigido"):
            Kv = cargar_version(aristas=aristas, nodos=nodos, ambito="espana", peso="km", general=general)
            m = medir(vecino_mas_cercano, Kv, inicio="Madrid")       # tiempo y coste

            # La memoria se mide aparte en KB: en MB saldría 0,00 (el algoritmo solo guarda listas de n ciudades)
            tracemalloc.start()
            vecino_mas_cercano(Kv, inicio="Madrid")
            _, pico = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            print(f"{aristas:>21} | {'km':>4} | {general:>24} | {m['ciudades']:>8} | "
                  f"{m['coste']:>10.1f} | {m['segundos'] * 1000:>6.2f} ms | {pico / 1e3:>6.1f} KB")

# El peso solo cambia los números (segundos en vez de km), no las operaciones del algoritmo
print()
for peso in ("km", "tiempo"):
    Kt = cargar_version(aristas="completo", nodos="capitales_y_100k", ambito="espana", peso=peso, general="grafo_dirigido")
    m = medir(vecino_mas_cercano, Kt, inicio="Madrid")
    print(f"peso={peso:>6}: {m['ciudades']} ciudades, coste {m['coste']:>10.1f} {m['unidad']}, "
          f"{m['segundos'] * 1000:.2f} ms")

              aristas | peso |                  general | ciudades |      coste |    tiempo |   memoria
             completo |   km |           grafo_dirigido |       47 |     6772.1 |   0.40 ms |    1.4 KB
             completo |   km | grafo_simple_no_dirigido |       47 |     6731.8 |   0.20 ms |    1.4 KB
             completo |   km |           grafo_dirigido |       75 |     8101.9 |   1.50 ms |    1.9 KB
             completo |   km | grafo_simple_no_dirigido |       75 |     7997.6 |   0.70 ms |    1.9 KB
 autovias_principales |   km |           grafo_dirigido |       47 |     7378.6 |   0.20 ms |    1.4 KB
 autovias_principales |   km | grafo_simple_no_dirigido |       47 |     7272.7 |   0.20 ms |    1.4 KB
 autovias_principales |   km |           grafo_dirigido |       75 |     8385.8 |   0.60 ms |    1.9 KB
 autovias_principales |   km | grafo_simple_no_dirigido |       75 |     8234.8 |   0.60 ms |    1.9 KB
         sin_autovias |   km |           grafo_dirigido |       

<div style="border-left:5px solid #55A868;color:#000;background:#EAF6ED;padding:10px 14px;margin:10px 0;border-radius:4px;">

**🧮 RESULTADOS**

**Prueba rápida (Galicia, 6 ciudades, desde Vigo).** Ruta devuelta:

> **Vigo → Pontevedra → Santiago de Compostela → A Coruña → Lugo → Ourense → Vigo = 465,8 km**

frente al **óptimo real de 460,4 km** (el mejor de los `6! = 720` ciclos): se queda a **+5,4 km (+1,2 %)**. `validar_ruta` confirma que visita cada ciudad una vez y `medir` que el coste devuelto coincide con el de la ruta.

**Calidad frente al óptimo y efecto del arranque** (Andalucía, subgrafos de las `n` primeras ciudades, `completo`/`km`/dirigido; para cada `n` se ejecuta el voraz desde **cada** ciudad):

| Ciudades | Óptimo | Mejor voraz | Peor voraz | Gap del peor |
|---:|---:|---:|---:|---:|
| 5 | 991,3 km | 991,3 km | 1.281,5 km | +29,3 % |
| 6 | 1.006,4 km | 1.006,4 km | 1.074,6 km | +6,8 % |
| 7 | 1.197,6 km | 1.197,6 km | 1.483,3 km | +23,9 % |
| 8 | 1.231,4 km | 1.231,4 km | 1.777,2 km | +44,3 % |
| 9 | 1.235,1 km | 1.235,1 km | 1.782,9 km | +44,4 % |

El mejor arranque llega al óptimo en estas cinco instancias, pero el peor se aleja hasta un **44 %**: el voraz puede ser excelente o malo según la primera decisión, y no hay forma de saberlo sin probar. En Galicia, arrancar en A Coruña, Lugo o Pontevedra da 460,4 km (el óptimo) y en Ourense o Santiago, 488,7 km (**+6,1 %**). En las 75 ciudades, el resultado va de **6.965 km** (Córdoba) a **8.641 km** (León), con **8.102 km** por defecto y **7.706 km** de media: un **24 %** entre el mejor y el peor arranque.

**Escalabilidad** (España, `km`, `grafo_dirigido`, arrancando en Madrid; la tabla completa con las 16 combinaciones está en la salida de la celda):

| Carreteras | Ciudades | Coste | Tiempo | Memoria pico |
|---|---:|---:|---:|---:|
| completo | 47 | 6.772,1 km | 0,20 ms | 1,4 KB |
| completo | 75 | 8.101,9 km | 0,70 ms | 1,9 KB |
| autovias_principales | 75 | 8.385,8 km | 0,50 ms | 1,9 KB |
| sin_autovias | 75 | 11.080,5 km | 0,50 ms | 1,9 KB |
| solo_autovias | 44 | 9.901,2 km | 0,20 ms | 1,3 KB |
| solo_autovias | 72 | 10.168,5 km | 0,50 ms | 1,9 KB |

**Conclusiones.**

- **Funciona con todas las ciudades** del trabajo (47 y 75) en **menos de un milisegundo** y con **menos de 2 KB**: el coste es `O(n²)` y la memoria `O(n)`, como predice la teoría. A diferencia de la fuerza bruta, el tamaño no es un problema.
- **El tiempo no es el límite; la calidad sí.** No hay ninguna garantía de optimalidad y el resultado depende mucho del arranque: el mejor de los probados llega al óptimo en las instancias pequeñas, pero el peor se queda un 44 % por encima.
- **Es determinista**: dos ejecuciones dan la misma ruta y el mismo coste, gracias al desempate por orden alfabético.
- **No necesita simetría**: funciona en dirigido y no dirigido. Las diferencias entre ambos (p. ej. `sin_autovias` con 47 ciudades: 10.913,8 km dirigido frente a 9.650,1 km no dirigido) vienen de los sentidos únicos, no del algoritmo.
- **El peso es indiferente**: con `tiempo` hace las mismas operaciones y devuelve segundos (288.035,7 s ≈ 80 h para las 75 ciudades).
- **Mejoras naturales:** probar varios arranques y quedarse con el mejor (multiarranque, `O(n³)`, sigue siendo instantáneo con 75 ciudades) y aplicar después **2-opt** para eliminar cruces. El vecino más cercano es, sobre todo, una **solución inicial excelente** para esos métodos.
- La memoria de `medir` sale redondeada a 0,00 MB porque el algoritmo solo guarda listas de `n` ciudades; por eso el pico se mide aparte, en KB.

</div>

---